# Vector Search on Your Existing Embeddings Table (psycopg2 edition)

Same connection style as your `sql_query_expert_v5` notebook — a `DB_CONFIG` dict with
`psycopg2` — so you can **copy your working values straight across**.

**What it does:**

1. **Inspect** your table so you know its exact structure
2. **Embed** the user's question with `text-embedding-004` (the SAME model that made your stored vectors)
3. **Search** with pgvector's cosine-distance operator `<=>`
4. **Answer** — Gemini turns the retrieved chunks into a proper natural-language answer

Run every cell top-to-bottom, in order.

## Step 1 — Install the libraries

In [ ]:
%pip install -q google-genai psycopg2-binary pandas

print("Libraries installed. If pip printed a 'restart kernel' notice, restart the kernel now.")

## Step 2 — Configuration

**Copy `DB_CONFIG` from your sql_query_expert_v5 notebook** — same host, port, dbname, user, password.
(If host is `localhost` there because you run the Cloud SQL Auth Proxy, keep it `localhost` here too —
just make sure the proxy is running.)

For the three `EXISTING TABLE` values, make a best guess — **Step 4 will show you the real names**.

In [ ]:
# ---------- Google Cloud / Vertex AI (same as your other notebook) ----------
PROJECT_ID = "your-gcp-project-id"     # CHANGE
LOCATION   = "us-central1"
EMBEDDING_MODEL = "text-embedding-004" # MUST match the model that made your stored embeddings
CHAT_MODEL      = "gemini-2.5-flash"

# ---------- PostgreSQL (copy from sql_query_expert_v5) ----------
DB_CONFIG = {
    "host":     "localhost",           # CHANGE if needed
    "port":     5432,
    "dbname":   "your_database",       # CHANGE
    "user":     "postgres",            # CHANGE
    "password": "your_password",       # CHANGE
}

# ---------- YOUR EXISTING TABLE (Step 4 helps you verify these) ----------
TABLE_NAME       = "your_table_name"   # table that holds the embeddings
CONTENT_COLUMN   = "content"           # column with the document text
EMBEDDING_COLUMN = "embedding"         # column with the vectors

TOP_K = 5   # how many chunks to retrieve per question

print("Configuration loaded (checked for real in the next steps).")

## Step 3 — Connect to PostgreSQL (same pattern as your other notebook)

In [ ]:
import pandas as pd
import psycopg2
import psycopg2.extras

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

try:
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute("SELECT current_database(), version();")
        dbname, version = cur.fetchone()
    print(f"Connected to database '{dbname}'")
    print(f"Server: {version.split(',')[0]}")
except Exception as e:
    raise SystemExit(f"Could not connect to PostgreSQL: {e}\n"
                     "  -> Check host/port/dbname/user/password in Step 2.\n"
                     "  -> If host is 'localhost', make sure the Cloud SQL Auth Proxy is running.")

## Step 4 — Inspect your embeddings table

First: find every table that has a `vector` (embedding) column — one of them is yours.
Then set `TABLE_NAME` / `EMBEDDING_COLUMN` in Step 2 to match and re-run that cell.

In [ ]:
with get_conn() as conn, conn.cursor() as cur:
    cur.execute("""
        SELECT table_name, column_name
        FROM information_schema.columns
        WHERE table_schema = 'public' AND udt_name = 'vector'
    """)
    rows = cur.fetchall()

if rows:
    print("Tables with a vector (embedding) column:")
    for t, c in rows:
        print(f"  table: {t:30s} embedding column: {c}")
    print("\n-> Set TABLE_NAME and EMBEDDING_COLUMN in Step 2 to match, then re-run Step 2.")
else:
    print("No vector columns found — is pgvector enabled, and is DB_CONFIG pointing at the right database?")

Now check the columns, row count, and vector dimension of YOUR table. `text-embedding-004` produces **768** dimensions — the printed dimension must say 768, otherwise the stored vectors were made by a different model.

In [ ]:
with get_conn() as conn, conn.cursor() as cur:
    cur.execute("""
        SELECT column_name, udt_name FROM information_schema.columns
        WHERE table_schema = 'public' AND table_name = %s
        ORDER BY ordinal_position
    """, (TABLE_NAME,))
    print(f"Columns of '{TABLE_NAME}':")
    for name, typ in cur.fetchall():
        print(f"  {name:25s} {typ}")

    cur.execute(f'SELECT COUNT(*) FROM {TABLE_NAME}')
    print(f"\nRows: {cur.fetchone()[0]}")

    cur.execute(f'SELECT vector_dims({EMBEDDING_COLUMN}) FROM {TABLE_NAME} LIMIT 1')
    dim = cur.fetchone()[0]
    print(f"Vector dimension: {dim}  (should be 768 for text-embedding-004)")

    cur.execute(f'SELECT {CONTENT_COLUMN} FROM {TABLE_NAME} LIMIT 1')
    print(f"\nSample {CONTENT_COLUMN}: {str(cur.fetchone()[0])[:200]}")

## Step 5 — Connect to Vertex AI & embed questions

**The golden rule of vector search:** the question must be embedded by the *same model*
(`text-embedding-004`) that embedded your documents. Different models produce vectors in
different coordinate systems — mixing them gives garbage results.

`task_type="RETRIEVAL_QUERY"` tells the model this is a search question, not a document.

In [ ]:
from google import genai
from google.genai.types import EmbedContentConfig

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

def embed_question(question: str) -> list[float]:
    resp = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=[question],
        config=EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    return resp.embeddings[0].values

v = embed_question("test question")
print(f"Vertex AI works — got a {len(v)}-dimensional vector (must match Step 4's dimension).")

## Step 6 — Vector search

`<=>` is pgvector's **cosine distance**: 0 = identical meaning, ~1 = unrelated.
Postgres sorts every row by distance to the question's vector and returns the closest `TOP_K`.

In [ ]:
def search(question: str, top_k: int = TOP_K) -> list[dict]:
    qvec = embed_question(question)

    with get_conn() as conn, conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        cur.execute(f"""
            SELECT {CONTENT_COLUMN} AS content,
                   {EMBEDDING_COLUMN} <=> %s::vector AS distance
            FROM {TABLE_NAME}
            ORDER BY distance
            LIMIT %s
        """, (str(qvec), top_k))
        rows = cur.fetchall()

    return [{"content": r["content"], "distance": float(r["distance"])} for r in rows]

# Development check: see WHAT gets retrieved (users never see this — they get Step 7's answer)
for i, r in enumerate(search("PUT A QUESTION ABOUT YOUR DOCUMENTS HERE", top_k=3), 1):
    print(f"--- Result {i}  (distance: {r['distance']:.4f}) ---")
    print(r["content"][:250], "\n")

## Step 7 — `ask()`: the function your users actually call

Retrieves the relevant chunks, hands them to Gemini as context, returns a **proper
natural-language answer** — no vectors, no chunks, no distances.

In [ ]:
def ask(question: str, top_k: int = TOP_K) -> str:
    hits = search(question, top_k)
    context = "\n\n---\n\n".join(h["content"] for h in hits)

    prompt = f"""Answer the question using ONLY the context below.
If the context doesn't contain the answer, say you don't know.

CONTEXT:
{context}

QUESTION: {question}"""

    resp = client.models.generate_content(model=CHAT_MODEL, contents=prompt)
    return resp.text

print("Pipeline ready — use ask(\"your question\") below.")

## Step 8 — Try it

In [ ]:
print(ask("PUT A QUESTION ABOUT YOUR DOCUMENTS HERE"))

## Troubleshooting

| Symptom | Fix |
|---|---|
| `could not connect to server` | DB host/port wrong, or the Cloud SQL Auth Proxy isn't running (Step 2/3) |
| `403 PERMISSION_DENIED` from Vertex | `gcloud auth application-default login` + enable the Vertex AI API |
| Wrong/irrelevant chunks retrieved | confirm the stored vectors really came from text-embedding-004; also try `task_type=None` in `embed_question` |
| `vector_dims` error or dimension is not 768 | embedding column isn't a pgvector `vector` type, or a different model made the vectors — paste Step 4's output to me |
| Searches slow on a big table | run once: `CREATE INDEX ON your_table USING hnsw (embedding vector_cosine_ops);` |
| Good chunks but bad answers | raise TOP_K, or improve the prompt in Step 7 |